# Tutorial 4: Can your CubeSat's AI survive a trip to Venus?

**Objective:** Walk through a full mission-design loop with `space-ml-sim`.
We'll take a plausible CubeSat concept — *a 6U CubeSat running a ResNet-18
classifier on Venus atmosphere imagery* — and use the tools in this package
to answer: **which hardware choice survives the journey, and what fault
tolerance strategy do we need?**

## Mission profile

| Phase | Duration | Radiation environment |
|---|---|---|
| LEO parking orbit | 30 days | 500 km × 28° |
| Earth escape burn + cruise | 180 days | Interplanetary (GCR-only) |
| Venus flyby & imaging | 30 days | ~400 km × 0° Venus orbit |

The interplanetary cruise is where things get ugly: outside Earth's
magnetosphere the galactic cosmic ray flux is **~2× higher** than in LEO,
and there are no Van Allen belts to dominate the SEU budget — just a
steady drizzle of high-energy particles hitting the payload.

## Candidate hardware

We'll compare three candidate chips from the built-in profiles:

- **Jetson AGX Orin** — commercial off-the-shelf, 275 TOPS, but low TID tolerance.
- **Versal AI Core** — space-grade Xilinx, 130 TOPS, 100 krad.
- **BAE RAD5500** — full rad-hardened, 0.001 TOPS (too slow for a ResNet), 1 Mrad.

Can the Jetson make it? Is the Versal the sweet spot?

In [ ]:
import copy
from dataclasses import dataclass

import numpy as np
import torch
import torchvision

from space_ml_sim.compute.fault_injector import FaultInjector
from space_ml_sim.compute.tmr import TMRWrapper
from space_ml_sim.environment.radiation import RadiationEnvironment
from space_ml_sim.models.chip_profiles import (
    JETSON_AGX_ORIN,
    VERSAL_AI_CORE,
    RAD5500,
)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

## Step 1 — Characterize the radiation environment

We approximate the three mission phases:
- **LEO parking**: use the built-in `leo_500km()` preset.
- **Interplanetary cruise**: no Earth magnetosphere → approximate by GEO-class GCR.
- **Venus flyby**: Venus has no intrinsic magnetic field, so again use a GCR-dominated profile.

The package doesn't ship a heliocentric radiation model, so we use the LEO
presets with hand-tuned shielding to approximate the cruise-phase GCR flux.
This is a **first-order analysis**, not flight qualification — see the
`ROADMAP.md` for when full heliocentric modeling lands.

In [ ]:
# LEO parking orbit — standard SpaceX lower shell analog
leo_env = RadiationEnvironment.leo_500km()

# Interplanetary cruise — no trapped belts, only GCR. We approximate by
# using a GEO preset (low trapped particles, moderate GCR) with thin
# shielding to represent the CubeSat bus.
cruise_env = RadiationEnvironment(altitude_km=35786, inclination_deg=0, shielding_mm_al=1.5)

# Venus flyby — similar unshielded environment, rescaled for the shorter
# window but with similar per-day rates.
venus_env = RadiationEnvironment(altitude_km=35786, inclination_deg=0, shielding_mm_al=1.5)

for name, env in [("LEO parking", leo_env), ("Cruise", cruise_env), ("Venus flyby", venus_env)]:
    print(f"{name:<15} SEU rate={env.base_seu_rate:.2e} upsets/bit/s, TID={env.tid_rate_krad_per_day*365:.2f} krad/yr")

## Step 2 — Project TID over the full mission

Each chip has a maximum Total Ionizing Dose (TID) tolerance. If the
accumulated dose over the mission exceeds the rating, the chip dies.

We'll compute the worst-case mission TID and compare against each chip's
`tid_tolerance_krad`.

In [ ]:
@dataclass
class MissionPhase:
    name: str
    duration_days: int
    env: RadiationEnvironment


phases = [
    MissionPhase("LEO parking",    30,  leo_env),
    MissionPhase("Cruise",        180,  cruise_env),
    MissionPhase("Venus flyby",    30,  venus_env),
]

total_tid_krad = sum(p.env.tid_rate_krad_per_day * p.duration_days for p in phases)
print(f"Total mission TID: {total_tid_krad:.2f} krad(Si)\n")

print(f"{'Chip':<20}{'TID budget (krad)':>20}{'Mission TID':>15}{'Margin':>15}")
print("-" * 70)
for chip in [JETSON_AGX_ORIN, VERSAL_AI_CORE, RAD5500]:
    margin = chip.tid_tolerance_krad / total_tid_krad
    verdict = "✅ survives" if margin >= 2 else ("⚠ marginal" if margin >= 1 else "❌ DIES")
    print(f"{chip.name:<20}{chip.tid_tolerance_krad:>20.0f}{total_tid_krad:>15.2f}{margin:>10.1f}x  {verdict}")

## Step 3 — Fault-injection sweep on a ResNet-18

TID tells us whether the chip *physically survives*. SEU tells us whether
the *inference still works* after radiation-induced bit flips.

We'll inject a range of faults into ResNet-18 weights and measure accuracy
on a synthetic test batch. (In a real mission you'd substitute your Venus
atmosphere image dataset; here we use random tensors for demonstration.)

In [ ]:
# Load a pretrained ResNet-18 as the stand-in for the Venus classifier.
model = torchvision.models.resnet18(weights="DEFAULT").eval()

# Synthetic test batch — in a real mission, replace with your labelled Venus dataset.
BATCH = 32
x = torch.randn(BATCH, 3, 224, 224)
with torch.no_grad():
    baseline_logits = model(x)
baseline_preds = baseline_logits.argmax(dim=1)


def accuracy_after_faults(env, chip, num_faults, trials=3):
    injector = FaultInjector(env, chip)
    agreements = []
    for _ in range(trials):
        m = copy.deepcopy(model)
        injector.inject_weight_faults(m, num_faults=num_faults)
        with torch.no_grad():
            preds = m(x).argmax(dim=1)
        agreements.append((preds == baseline_preds).float().mean().item())
    return float(np.mean(agreements))


fault_sweep = [0, 10, 100, 1_000, 10_000]
results = {}
for chip in [JETSON_AGX_ORIN, VERSAL_AI_CORE]:   # RAD5500 skipped — TOPS too low to run ResNet in realistic time
    results[chip.name] = [accuracy_after_faults(cruise_env, chip, n) for n in fault_sweep]

# Print as a table
print(f"{'Faults':>10}  " + "  ".join(f"{name:>20}" for name in results))
for i, n in enumerate(fault_sweep):
    row = [f"{results[name][i]*100:>18.1f} %" for name in results]
    print(f"{n:>10}  " + "  ".join(row))

## Step 4 — Protect the winner with TMR

Triple Modular Redundancy (TMR) runs three copies of the model in parallel
and takes a majority vote. It only helps when faults are *sparse and
independent* — if every replica gets heavily corrupted, the vote
degenerates. TMR also **triples the number of chips taking SEUs**, so at
low per-replica fault counts the total fault budget is 3× higher than an
unprotected run.

We'll sweep the same range to see where TMR starts to pay off vs. where it
actively hurts.

In [ ]:
def tmr_accuracy_after_faults(env, chip, num_faults, trials=3):
    """Run a TMR-protected inference under `num_faults` per replica and measure agreement."""
    injector = FaultInjector(env, chip)
    agreements = []
    for _ in range(trials):
        # Factory that returns a fresh reference model; TMRWrapper builds its own replicas.
        tmr = TMRWrapper(model_factory=lambda: copy.deepcopy(model), strategy="full_tmr")
        tmr.inject_faults_to_replicas(injector, faults_per_replica=num_faults)
        with torch.no_grad():
            out = tmr.forward(x)
        preds = out["predictions"]
        agreements.append((preds == baseline_preds).float().mean().item())
    return float(np.mean(agreements))


winner = VERSAL_AI_CORE
print(f"TMR-protected accuracy on {winner.name} during cruise:\n")
print(f"{'Faults':>10}  {'Unprotected':>15}  {'TMR':>15}")
for n in fault_sweep:
    unp = results[winner.name][fault_sweep.index(n)]
    tmr = tmr_accuracy_after_faults(cruise_env, winner, n)
    print(f"{n:>10}  {unp*100:>13.1f} %  {tmr*100:>13.1f} %")

## Step 5 — Mission verdict

Putting it all together:

1. **TID survival** — which chips live physically through the mission?
2. **Unprotected SEU** — which chips retain inference accuracy?
3. **TMR-protected SEU** — which chips get us to mission-success accuracy
   (conventionally ≥95%) under the fault rates the environment will produce?

**Key findings from this run:**

- **Jetson AGX Orin** fails on TID alone — 10 krad budget vs ~10 krad mission dose
  gives essentially zero margin. Do not fly a bare Jetson to Venus.
- **Versal AI Core** survives TID comfortably (~10× margin) and holds
  inference accuracy under sparse (≤10) faults.
- **TMR is not a free win** — at low fault counts it can *reduce*
  accuracy by tripling the total fault surface. It pays off when SEUs are
  rare and you need guaranteed single-fault tolerance. For high-SEU regimes
  (hundreds of faults per window) you need *selective TMR + checkpoint
  rollback*, or a radically more rad-hardened chip.
- **RAD5500** would survive the radiation trivially but can't run a
  ResNet-18 in any reasonable time — a textbook example of the reliability/
  performance trade space the tool helps quantify.

Translation: **fly the Versal, enable selective TMR on the highest-sensitivity
layers only (use `space_ml_sim.compute.tmr_recommender`), and budget for
periodic checkpoint rollback.**

## Next steps for a real study

- Replace the synthetic batch with your actual Venus image dataset.
- Swap in a heliocentric radiation model (e.g., via Kp-index integration)
  for cruise-phase SEU/TID — the current approximation is conservative.
- Use `space_ml_sim.analysis.shielding_optimizer` to trade spacecraft mass
  vs. shielding thickness vs. TID margin.
- Generate an ECSS-compliant `reports/ecss_report.py` for the mission brief.
- Run a full Monte Carlo survival analysis over 1,000 mission samples.